8mins 30.7 secs

## Libraries

In [25]:
import numpy as np
import pandas as pd
import time
import random

from sklearn.model_selection import ParameterGrid

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.neighbors import NearestNeighbors

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils import clip_grad_norm_

import os
import json
from pathlib import Path
import time
import uuid

In [26]:
print("Torch version:", torch.__version__)
print("CUDA (in torch):", torch.version.cuda)
print("cuda.is_available:", torch.cuda.is_available())
print("device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    print("capability:", torch.cuda.get_device_capability(0))

Torch version: 2.10.0.dev20251203+cu128
CUDA (in torch): 12.8
cuda.is_available: True
device count: 1
device: NVIDIA GeForce RTX 5070 Ti
capability: (12, 0)


## Config

In [27]:
DATA_PATH = "../../data/full_data.xlsx"

TIME_COL   = "Date"
TARGET_COL = "AveragePrice"
ENTITY_COL = "AreaCode"

TUNE_START_DATE = pd.Timestamp("2007-04-01")
TUNE_END_DATE   = pd.Timestamp("2022-03-31")

ROLLING_TRAIN_WINDOW = 90
ROLLING_VAL_WINDOW   = 18

param_grid = list(ParameterGrid({
    "WINDOW":       [12, 18],
    "HIDDEN_DIM":   [64, 128, 256],   # widened
    "DROPOUT":      [0.0, 0.2, 0.3],
    "LR":           [1e-3, 5e-4, 3e-4, 1e-4],  # widened
    "WEIGHT_DECAY": [0.0, 1e-4],

    # Two-channel graph settings
    "GATE_INIT":    [0.85],
    "K_DIST":       [8],
    "SIGMA_KM":     [None, 60.0],  # binary or exp(-d/sigma)
    "K_CORR":       [8],

    # Huber settings
    "HUBER_BETA":   [1.0],  # SmoothL1 beta; 1.0 is typical
}))

BATCH_SIZE    = 32
MAX_EPOCHS    = 120   # widened
PATIENCE      = 12    # widened
MAX_GRAD_NORM = 5.0

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Scheduler: Reduce LR on plateau (often helps these models)
USE_SCHEDULER = True
SCHED_FACTOR  = 0.5
SCHED_PATIENCE = 3
MIN_LR = 1e-6

print(f"Number of hyperparameter configs: {len(param_grid)}")

# ----------------------------
# Feature lists
# ----------------------------
continuous_cols = [
    # "AvgNeighbourPrice_lag1",
    # "local_I_lag1",
    "area_km2",
    "centroid_x",
    "centroid_y",
    "CoL_distance_km",
    "LA_FE",
    "sdlt_perc_threshold",
    "dwelling_stock_per_1000",
    "population",
    "ashe_weekly",
    "base_rate",
    "claimant_count_prop",
    "planning_decisions_per_1000",
    "planning_granted_prop",
    "rail_station_entry_exit",
    "GDP",
    "CPIH",
    "LA_embed_0",
    "LA_embed_1",
    "LA_embed_2",
    "LA_embed_3",
    "LA_embed_4"
]

categorical_cols = [
    # "LMIQuadrantlag1_2.0",
    # "LMIQuadrantlag1_3.0",
    # "LMIQuadrantlag1_4.0",
    "Region_East of England",
    "Region_London",
    "Region_North East",
    "Region_North West",
    "Region_South East",
    "Region_South West",
    "Region_West Midlands",
    "Region_Yorkshire and The Humber",
]

base_feature_cols = continuous_cols + categorical_cols

# ----------------------------
# Reproducibility
# ----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

Number of hyperparameter configs: 288


In [28]:
DEVICE

device(type='cuda')

## Crash saving

In [29]:
# ----------------------------
# Crash-safe saving (Windows-safe)
# ----------------------------
CHECKPOINT_DIR = Path("../../checkpoints/tgcn_tuning")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

PROGRESS_PATH  = CHECKPOINT_DIR / "progress.json"
RESULTS_PATH   = Path("../../results/tgcn_c1_gated_improved_rollingcv_partial.csv")
FOLD_RESULTS_PATH = CHECKPOINT_DIR / "tgcn_fold_results.csv"

def _rng_state():
    state = {
        "python_random_state": random.getstate(),
        "numpy_random_state": np.random.get_state(),
        "torch_random_state": torch.random.get_rng_state(),
        "cuda_random_state": None
    }
    if torch.cuda.is_available():
        state["cuda_random_state"] = torch.cuda.get_rng_state_all()
    return state

def _to_byte_tensor(x):
    if x is None:
        return None
    if isinstance(x, torch.Tensor):
        return x.to(dtype=torch.uint8, device="cpu")
    if isinstance(x, (bytes, bytearray)):
        return torch.tensor(list(x), dtype=torch.uint8)
    arr = np.asarray(x, dtype=np.uint8)
    return torch.from_numpy(arr)

def _set_rng_state(state):
    if state is None:
        return
    random.setstate(state["python_random_state"])
    np.random.set_state(state["numpy_random_state"])

    torch_state = _to_byte_tensor(state.get("torch_random_state"))
    if torch_state is not None:
        torch.random.set_rng_state(torch_state)

    if torch.cuda.is_available() and state.get("cuda_random_state") is not None:
        cuda_states = [_to_byte_tensor(s) for s in state["cuda_random_state"]]
        torch.cuda.set_rng_state_all(cuda_states)

def save_progress(cfg_id, fold_no, epoch, extra=None):
    payload = {"cfg_id": int(cfg_id), "fold_no": int(fold_no), "epoch": int(epoch)}
    if extra:
        payload.update(extra)

    # Windows-safe overwrite
    with open(PROGRESS_PATH, "w") as f:
        json.dump(payload, f, indent=2, default=str)
        f.flush()
        os.fsync(f.fileno())

def load_progress():
    if not PROGRESS_PATH.exists():
        return None
    with open(PROGRESS_PATH, "r") as f:
        return json.load(f)

def checkpoint_path(cfg_id, fold_no):
    return CHECKPOINT_DIR / f"ckpt_cfg{cfg_id:04d}_fold{fold_no:03d}.pt"

def save_checkpoint(
    cfg_id, fold_no, epoch,
    model, optimizer, scheduler,
    best_val, best_epoch, epochs_no_improve,
    best_state, extra=None
):
    ckpt = {
        "cfg_id": int(cfg_id),
        "fold_no": int(fold_no),
        "epoch": int(epoch),
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": (scheduler.state_dict() if scheduler is not None else None),
        "best_val": float(best_val),
        "best_epoch": int(best_epoch),
        "epochs_no_improve": int(epochs_no_improve),
        "best_state": best_state,
        "rng_state": _rng_state(),
        "extra": extra or {},
    }

    path = checkpoint_path(cfg_id, fold_no)
    tmp = path.with_suffix(path.suffix + f".{uuid.uuid4().hex}.tmp")
    torch.save(ckpt, tmp)

    for _ in range(20):
        try:
            os.replace(tmp, path)
            return
        except PermissionError:
            time.sleep(0.1)

    # fallback (non-atomic)
    torch.save(ckpt, path)
    try:
        if tmp.exists():
            tmp.unlink()
    except Exception:
        pass

def load_checkpoint(cfg_id, fold_no, device):
    path = checkpoint_path(cfg_id, fold_no)
    if not path.exists():
        return None
    return torch.load(path, map_location=device, weights_only=False)

def flush_results_partial(results_list):
    """Writes partial results and dedupes by cfg_id (keep last). Windows-safe."""
    if not results_list:
        return
    dfp = pd.DataFrame(results_list)
    if "cfg_id" in dfp.columns:
        dfp = dfp.drop_duplicates(subset=["cfg_id"], keep="last")

    tmp = RESULTS_PATH.with_suffix(RESULTS_PATH.suffix + f".{uuid.uuid4().hex}.tmp")
    dfp.to_csv(tmp, index=False)

    for _ in range(20):
        try:
            os.replace(tmp, RESULTS_PATH)
            return
        except PermissionError:
            time.sleep(0.1)

    dfp.to_csv(RESULTS_PATH, index=False)
    try:
        if tmp.exists():
            tmp.unlink()
    except Exception:
        pass

def append_fold_result(row: dict, done_pairs: set[tuple[int, int]]):
    """Append fold row once per (cfg_id, fold_no)."""
    key = (int(row["cfg_id"]), int(row["fold_no"]))
    if key in done_pairs:
        return

    df = pd.DataFrame([row])
    header = not FOLD_RESULTS_PATH.exists()
    df.to_csv(FOLD_RESULTS_PATH, mode="a", header=header, index=False)
    done_pairs.add(key)


## Metric functions

In [30]:
def mae(y, yhat):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return float(np.mean(np.abs(y - yhat)))

def rmse(y, yhat):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return float(np.sqrt(np.mean((y - yhat) ** 2)))

def smape(y, yhat, eps=1e-8):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return float(
        100.0 * np.mean(
            2.0 * np.abs(yhat - y) / (np.abs(y) + np.abs(yhat) + eps)
        )
    )

def mase(y, yhat, y_train, m=12, eps=1e-8):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    y_train = np.asarray(y_train)

    if len(y_train) <= m:
        return np.nan

    naive_diff = np.abs(y_train[m:] - y_train[:-m])
    scale = np.mean(naive_diff) + eps
    return float(np.mean(np.abs(y - yhat)) / scale)


## Load data

In [31]:
df = pd.read_excel(DATA_PATH, parse_dates=[TIME_COL])
df = df.sort_values([TIME_COL, ENTITY_COL]).reset_index(drop=True)

mask_tune = (df[TIME_COL] >= TUNE_START_DATE) & (df[TIME_COL] <= TUNE_END_DATE)
df = df.loc[mask_tune].copy()

df = (
    df.sort_values([TIME_COL, ENTITY_COL])
      .drop_duplicates(subset=[TIME_COL, ENTITY_COL], keep="last")
      .copy()
)

dates = pd.Index(sorted(df[TIME_COL].unique()))
T_total = len(dates)
la_order = sorted(df[ENTITY_COL].unique())
N = len(la_order)

print(f"Tuning period: {dates[0].date()} → {dates[-1].date()}")
print("Total months:", T_total)
print("Number of LAs:", N)

full_index = pd.MultiIndex.from_product([dates, la_order], names=[TIME_COL, ENTITY_COL])
feature_cols = base_feature_cols.copy()

df_panel = (
    df.set_index([TIME_COL, ENTITY_COL])
      .reindex(full_index)
      .sort_index()
)

# Forward-fill FEATURES only (never bfill). Leave TARGET missing if missing.
df_panel[feature_cols] = (
    df_panel[feature_cols]
      .groupby(level=ENTITY_COL)
      .ffill()
)

# Add price lags (do not bfill)
df_panel["price_lag1"]  = df_panel.groupby(level=ENTITY_COL)[TARGET_COL].shift(1)
df_panel["price_lag12"] = df_panel.groupby(level=ENTITY_COL)[TARGET_COL].shift(12)

# Forward-fill lag features only
df_panel[["price_lag1", "price_lag12"]] = (
    df_panel[["price_lag1", "price_lag12"]]
      .groupby(level=ENTITY_COL)
      .ffill()
)

lag_price_cols = ["price_lag1", "price_lag12"]
feature_cols = feature_cols + lag_price_cols
F = len(feature_cols)

X_all = df_panel[feature_cols].to_numpy(dtype=np.float32).reshape(T_total, N, F)
y_all = df_panel[TARGET_COL].to_numpy(dtype=np.float32).reshape(T_total, N)
y_all_orig = y_all.copy()

print("X_all shape:", X_all.shape)
print("y_all shape:", y_all.shape)

Tuning period: 2007-04-01 → 2022-03-01
Total months: 180
Number of LAs: 294
X_all shape: (180, 294, 31)
y_all shape: (180, 294)


## Rolling origin folds (time index space)

In [32]:
fold_specs = []
start_idx = ROLLING_TRAIN_WINDOW
while True:
    train_end_idx = start_idx
    val_start_idx = train_end_idx
    val_end_idx   = val_start_idx + ROLLING_VAL_WINDOW
    if val_end_idx > T_total:
        break
    train_start_idx = train_end_idx - ROLLING_TRAIN_WINDOW
    fold_specs.append((
        train_start_idx, train_end_idx, val_start_idx, val_end_idx,
        dates[train_start_idx], dates[train_end_idx - 1],
        dates[val_start_idx], dates[val_end_idx - 1],
    ))
    start_idx += ROLLING_VAL_WINDOW

print(f"Number of folds: {len(fold_specs)}")

Number of folds: 5


## Adjacency builders

In [33]:
def build_distance_knn_Ahat_from_panel(
    df_panel: pd.DataFrame,
    dates: pd.Index,
    la_order: list,
    k_dist: int = 8,
    sigma_km: float | None = None,
    eps: float = 1e-8,
) -> np.ndarray:
    """
    Uses centroid_x/centroid_y in EPSG:27700 (meters).
    Builds symmetric KNN adjacency; weights are either binary or exp(-d_km/sigma_km).
    Adds self-loops and returns degree-normalized A_hat (float32).
    """
    first_date = dates[0]
    cent = df_panel.loc[(first_date, la_order), ["centroid_x", "centroid_y"]].to_numpy(dtype=np.float32)

    nbrs = NearestNeighbors(n_neighbors=k_dist + 1, algorithm="auto").fit(cent)
    dists_m, idx = nbrs.kneighbors(cent)

    Nloc = len(la_order)
    A = np.zeros((Nloc, Nloc), dtype=np.float32)

    for i in range(Nloc):
        for n in range(1, k_dist + 1):
            j = int(idx[i, n])
            d_km = float(dists_m[i, n] / 1000.0)

            if sigma_km is None:
                w = 1.0
            else:
                w = float(np.exp(-d_km / (sigma_km + eps)))

            if w > A[i, j]:
                A[i, j] = w
            if w > A[j, i]:
                A[j, i] = w

    A = A + np.eye(Nloc, dtype=np.float32)
    deg = A.sum(axis=1)
    D_inv_sqrt = np.diag(1.0 / np.sqrt(deg + eps))
    A_hat = (D_inv_sqrt @ A @ D_inv_sqrt).astype(np.float32)
    return A_hat

def build_corr_knn_Ahat_train_only(
    y_train_TN: np.ndarray,
    k_corr: int = 8,
    eps: float = 1e-8,
) -> np.ndarray:
    """
    Train-only correlation graph. Fills NaNs with per-node TRAIN mean.
    """
    y = y_train_TN.copy()
    col_means = np.nanmean(y, axis=0)
    inds = np.where(np.isnan(y))
    if inds[0].size > 0:
        y[inds] = np.take(col_means, inds[1])

    corr = np.corrcoef(y.T)
    corr = np.nan_to_num(corr, nan=0.0, posinf=0.0, neginf=0.0)
    np.fill_diagonal(corr, 0.0)

    Nloc = corr.shape[0]
    A = np.zeros((Nloc, Nloc), dtype=np.float32)

    for i in range(Nloc):
        nbr_idx = np.argsort(-np.abs(corr[i]))[:k_corr]
        for j in nbr_idx:
            A[i, j] = 1.0
            A[j, i] = 1.0

    A = A + np.eye(Nloc, dtype=np.float32)
    deg = A.sum(axis=1)
    D_inv_sqrt = np.diag(1.0 / np.sqrt(deg + eps))
    A_hat = (D_inv_sqrt @ A @ D_inv_sqrt).astype(np.float32)
    return A_hat

## Dataset class (window will vary per config)

In [34]:
class SpatioTemporalDataset(Dataset):
    def __init__(self, X, y, window):
        self.X = X
        self.y = y
        self.window = window
        self.T, self.N, self.F = X.shape
        self.indices = list(range(window, self.T))

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        t = self.indices[idx]
        X_seq = self.X[t - self.window:t]  # [window, N, F]
        y_t   = self.y[t]                  # [N]
        return torch.tensor(X_seq, dtype=torch.float32), torch.tensor(y_t, dtype=torch.float32)


## LA scalers

In [35]:
class PerNodeRobustScaler:
    """
    Per-node robust scaling using median and IQR (approx robust).
    Stores per-node center and scale.
    """
    def __init__(self, eps: float = 1e-8):
        self.eps = eps
        self.center_ = None  # [N]
        self.scale_  = None  # [N]

    def fit(self, y_train_TN: np.ndarray):
        # y_train_TN: [T, N]
        y = y_train_TN.copy()
        # median per node
        med = np.nanmedian(y, axis=0)
        q1  = np.nanpercentile(y, 25, axis=0)
        q3  = np.nanpercentile(y, 75, axis=0)
        iqr = (q3 - q1)
        iqr = np.where(np.isfinite(iqr) & (iqr > self.eps), iqr, 1.0)  # avoid 0
        self.center_ = med.astype(np.float32)
        self.scale_  = iqr.astype(np.float32)
        return self

    def transform(self, y_TN: np.ndarray) -> np.ndarray:
        y = y_TN.astype(np.float32, copy=True)
        out = (y - self.center_[None, :]) / (self.scale_[None, :] + self.eps)
        # keep NaNs as NaNs
        out[np.isnan(y)] = np.nan
        return out

    def inverse_transform(self, y_TN: np.ndarray) -> np.ndarray:
        y = y_TN.astype(np.float32, copy=True)
        out = y * (self.scale_[None, :] + self.eps) + self.center_[None, :]
        out[np.isnan(y)] = np.nan
        return out
    
# ============================================================
# Masked Huber loss (supports missing targets)
# ============================================================
def masked_huber_loss(y_hat: torch.Tensor, y_true: torch.Tensor, beta: float = 1.0) -> torch.Tensor:
    """
    y_hat, y_true: [B, N]
    Computes SmoothL1 (Huber) on observed entries only.
    """
    mask = torch.isfinite(y_true)
    if mask.sum() == 0:
        # return a dummy finite loss; caller should skip backward if desired
        return torch.tensor(0.0, device=y_hat.device)
    diff = y_hat[mask] - y_true[mask]
    abs_diff = diff.abs()
    # SmoothL1 with beta:
    # if |d| < beta: 0.5 * d^2 / beta
    # else: |d| - 0.5*beta
    loss = torch.where(
        abs_diff < beta,
        0.5 * (diff * diff) / beta,
        abs_diff - 0.5 * beta
    )
    return loss.mean()

## Model

In [36]:
class GraphConv2GatedSeparate(nn.Module):
    """
    out = g * linear_dist(A_dist X) + (1-g) * linear_corr(A_corr X)
    Gate is learnable scalar.
    """
    def __init__(self, in_feats, out_feats, gate_init=0.85):
        super().__init__()
        self.linear_dist = nn.Linear(in_feats, out_feats)
        self.linear_corr = nn.Linear(in_feats, out_feats)

        gate_init = float(np.clip(gate_init, 1e-4, 1 - 1e-4))
        init_logit = np.log(gate_init / (1.0 - gate_init))
        self.gate_logit = nn.Parameter(torch.tensor(init_logit, dtype=torch.float32))

    def forward(self, X, A_dist, A_corr):
        AX_dist = torch.einsum("ij,bjf->bif", A_dist, X)
        AX_corr = torch.einsum("ij,bjf->bif", A_corr, X)

        out_dist = self.linear_dist(AX_dist)
        out_corr = self.linear_corr(AX_corr)

        g = torch.sigmoid(self.gate_logit)
        return g * out_dist + (1.0 - g) * out_corr

    def gate_value(self) -> float:
        return float(torch.sigmoid(self.gate_logit).detach().cpu().item())

class TGCNCell2(nn.Module):
    def __init__(self, in_feats, hidden_dim, dropout=0.0, gate_init=0.85):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.gc_zr = GraphConv2GatedSeparate(in_feats + hidden_dim, 2 * hidden_dim, gate_init=gate_init)
        self.gc_h  = GraphConv2GatedSeparate(in_feats + hidden_dim, hidden_dim,     gate_init=gate_init)
        self.dropout = nn.Dropout(dropout)

    def forward(self, X_t, H_prev, A_dist, A_corr):
        if H_prev is None:
            H_prev = torch.zeros(X_t.size(0), X_t.size(1), self.hidden_dim, device=X_t.device)

        XH = torch.cat([X_t, H_prev], dim=-1)
        ZR = torch.sigmoid(self.gc_zr(XH, A_dist, A_corr))
        Z, R = torch.chunk(ZR, 2, dim=-1)

        XH_candidate = torch.cat([X_t, R * H_prev], dim=-1)
        H_tilde = torch.tanh(self.gc_h(XH_candidate, A_dist, A_corr))

        H_new = (1 - Z) * H_prev + Z * H_tilde
        H_new = self.dropout(H_new)
        return H_new

class TGCN2(nn.Module):
    def __init__(self, in_feats, hidden_dim, dropout=0.0, gate_init=0.85):
        super().__init__()
        self.cell = TGCNCell2(in_feats, hidden_dim, dropout=dropout, gate_init=gate_init)
        self.out  = nn.Linear(hidden_dim, 1)

    def forward(self, X_seq, A_dist, A_corr):
        H = None
        for t in range(X_seq.size(1)):
            H = self.cell(X_seq[:, t], H, A_dist, A_corr)
        return self.out(H).squeeze(-1)  # [B, N]

## Train and Eval with early stopping

In [37]:
# --------------------------
# Train and Eval with early stopping (crash-safe)
# --------------------------

# Load existing partial config results (so results survive restarts)
if RESULTS_PATH.exists():
    results = pd.read_csv(RESULTS_PATH).to_dict("records")
else:
    results = []

# Resume info
progress = load_progress()
resume_cfg_id  = int(progress["cfg_id"]) if progress else 1
resume_fold_no = int(progress["fold_no"]) if progress else 1
resume_epoch   = int(progress["epoch"]) if progress else 1
print("Resume info:", progress)

# Load existing fold log once
fr = pd.read_csv(FOLD_RESULTS_PATH) if FOLD_RESULTS_PATH.exists() else pd.DataFrame()

# Build a fast "already written" set for folds (prevents duplicates even if you rerun)
done_pairs = set()
if not fr.empty:
    done_pairs = set(zip(fr["cfg_id"].astype(int).tolist(), fr["fold_no"].astype(int).tolist()))

for cfg_id, params in enumerate(param_grid, start=1):
    if cfg_id < resume_cfg_id:
        continue

    WINDOW      = params["WINDOW"]
    HIDDEN_DIM  = params["HIDDEN_DIM"]
    DROPOUT     = params["DROPOUT"]
    LR          = params["LR"]
    WD          = params["WEIGHT_DECAY"]
    GATE_INIT   = params["GATE_INIT"]
    K_DIST      = params["K_DIST"]
    SIGMA_KM    = params["SIGMA_KM"]
    K_CORR      = params["K_CORR"]
    HUBER_BETA  = params["HUBER_BETA"]

    print(f"\n=== Config {cfg_id}/{len(param_grid)} ===")
    print(params)

    # Static distance adjacency per config
    A_hat_dist_np = build_distance_knn_Ahat_from_panel(
        df_panel=df_panel,
        dates=dates,
        la_order=la_order,
        k_dist=K_DIST,
        sigma_km=SIGMA_KM,
    )
    A_dist = torch.tensor(A_hat_dist_np, dtype=torch.float32, device=DEVICE)

    # Preload completed fold metrics from fold log
    fold_mae_list, fold_rmse_list, fold_smape_list, fold_mase_list = [], [], [], []
    fold_count = 0
    done_folds = set()

    if not fr.empty:
        fr_cfg = fr[fr["cfg_id"] == cfg_id].sort_values("fold_no")
        if not fr_cfg.empty:
            fold_mae_list   = fr_cfg["MAE"].tolist()
            fold_rmse_list  = fr_cfg["RMSE"].tolist()
            fold_smape_list = fr_cfg["sMAPE"].tolist()
            fold_mase_list  = fr_cfg["MASE"].tolist()
            fold_count = len(fr_cfg)
            done_folds = set(fr_cfg["fold_no"].astype(int).tolist())

    for fold_no, (train_start_idx, train_end_idx,
                  val_start_idx, val_end_idx,
                  train_start_date, train_end_date,
                  val_start_date, val_end_date) in enumerate(fold_specs, start=1):

        # Skip if already logged
        if fold_no in done_folds:
            print(f"    (skip fold {fold_no}: already logged)")
            continue

        print(f"  Fold {fold_no}: Train {train_start_date:%Y-%m}–{train_end_date:%Y-%m}, "
              f"Val {val_start_date:%Y-%m}–{val_end_date:%Y-%m}")

        X_train = X_all[train_start_idx:train_end_idx]
        y_train = y_all[train_start_idx:train_end_idx]
        X_val   = X_all[val_start_idx:val_end_idx]
        y_val   = y_all[val_start_idx:val_end_idx]

        if X_train.shape[0] < WINDOW + 1 or X_val.shape[0] < 1:
            print("    (skip fold: not enough time steps)")
            continue

        # Fold-specific correlation adjacency (train-only)
        A_hat_corr_np = build_corr_knn_Ahat_train_only(y_train, k_corr=K_CORR)
        A_corr = torch.tensor(A_hat_corr_np, dtype=torch.float32, device=DEVICE)

        # Feature last-resort fill using TRAIN feature means (features only)
        if np.isnan(X_train).any():
            train_feat_means = np.nanmean(X_train.reshape(-1, F), axis=0)
            X_train = np.where(np.isnan(X_train), train_feat_means[None, None, :], X_train)
        if np.isnan(X_val).any():
            train_feat_means = np.nanmean(X_train.reshape(-1, F), axis=0)
            X_val = np.where(np.isnan(X_val), train_feat_means[None, None, :], X_val)

        # X scaling: train-only
        x_scaler = StandardScaler()
        X_train_scaled = x_scaler.fit_transform(X_train.reshape(-1, F)).reshape(X_train.shape).astype(np.float32)
        X_val_scaled   = x_scaler.transform(X_val.reshape(-1, F)).reshape(X_val.shape).astype(np.float32)

        # y scaling: per-node robust scaler (train-only)
        y_scaler = PerNodeRobustScaler().fit(y_train)
        y_train_scaled = y_scaler.transform(y_train).astype(np.float32)
        y_val_scaled   = y_scaler.transform(y_val).astype(np.float32)

        # Build VAL with TRAIN tail context
        X_context = np.concatenate([X_train_scaled[-WINDOW:], X_val_scaled], axis=0).astype(np.float32)
        y_context = np.concatenate([y_train_scaled[-WINDOW:], y_val_scaled], axis=0).astype(np.float32)

        train_ds = SpatioTemporalDataset(X_train_scaled, y_train_scaled, window=WINDOW)
        val_ds   = SpatioTemporalDataset(X_context,      y_context,      window=WINDOW)

        if len(train_ds) == 0 or len(val_ds) == 0:
            print("    (skip fold: empty dataset after windowing)")
            continue

        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
        val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)

        model = TGCN2(in_feats=F, hidden_dim=HIDDEN_DIM, dropout=DROPOUT, gate_init=GATE_INIT).to(DEVICE)
        optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WD)

        if USE_SCHEDULER:
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                optimizer, mode="min", factor=SCHED_FACTOR, patience=SCHED_PATIENCE, min_lr=MIN_LR
            )
        else:
            scheduler = None

        best_val = np.inf
        best_epoch = -1
        epochs_no_improve = 0
        best_state = None

        # Resume fold checkpoint if available
        start_epoch = 1
        ckpt = load_checkpoint(cfg_id, fold_no, DEVICE)
        if ckpt is not None:
            print(f"    🔁 Resuming from checkpoint: cfg {cfg_id}, fold {fold_no}, epoch {ckpt['epoch']}")
            model.load_state_dict(ckpt["model_state"])
            optimizer.load_state_dict(ckpt["optimizer_state"])
            if scheduler is not None and ckpt["scheduler_state"] is not None:
                scheduler.load_state_dict(ckpt["scheduler_state"])

            best_val = ckpt["best_val"]
            best_epoch = ckpt["best_epoch"]
            epochs_no_improve = ckpt["epochs_no_improve"]
            best_state = ckpt["best_state"]
            _set_rng_state(ckpt.get("rng_state"))
            start_epoch = int(ckpt["epoch"]) + 1

        # Honor progress.json mid-epoch info
        if progress and cfg_id == resume_cfg_id and fold_no == resume_fold_no:
            start_epoch = max(start_epoch, resume_epoch)

        for epoch in range(start_epoch, MAX_EPOCHS + 1):
            model.train()
            train_losses = []

            for X_seq, y_t in train_loader:
                X_seq = X_seq.to(DEVICE)
                y_t   = y_t.to(DEVICE)

                optimizer.zero_grad()
                y_hat = model(X_seq, A_dist, A_corr)
                loss = masked_huber_loss(y_hat, y_t, beta=HUBER_BETA)

                if (not torch.isfinite(loss)) or (loss.item() == 0.0 and (not torch.isfinite(y_t).any())):
                    continue

                loss.backward()
                clip_grad_norm_(model.parameters(), max_norm=MAX_GRAD_NORM)
                optimizer.step()
                train_losses.append(loss.item())

            if len(train_losses) == 0:
                print("    ⚠ No valid training batches. Skipping fold.")
                break

            model.eval()
            val_losses = []
            with torch.no_grad():
                for X_seq, y_t in val_loader:
                    X_seq = X_seq.to(DEVICE)
                    y_t   = y_t.to(DEVICE)
                    y_hat = model(X_seq, A_dist, A_corr)
                    vloss = masked_huber_loss(y_hat, y_t, beta=HUBER_BETA)
                    if torch.isfinite(vloss) and torch.isfinite(y_t).any():
                        val_losses.append(vloss.item())

            if len(val_losses) == 0:
                print("    ⚠ No valid validation batches. Skipping fold.")
                break

            val_loss = float(np.mean(val_losses))
            if scheduler is not None:
                scheduler.step(val_loss)

            lr_now = optimizer.param_groups[0]["lr"]
            g_zr = model.cell.gc_zr.gate_value()
            g_h  = model.cell.gc_h.gate_value()

            print(f"    Epoch {epoch:03d} | trainHuber={np.mean(train_losses):.4f} | "
                  f"valHuber={val_loss:.4f} | lr={lr_now:.1e} | gates(zr={g_zr:.3f}, h={g_h:.3f})")

            if val_loss + 1e-6 < best_val:
                best_val = val_loss
                best_epoch = epoch
                epochs_no_improve = 0
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= PATIENCE:
                    print(f"    Early stopping at epoch {epoch}")
                    save_checkpoint(cfg_id, fold_no, epoch, model, optimizer, scheduler,
                                    best_val, best_epoch, epochs_no_improve, best_state,
                                    extra={"params": params})
                    save_progress(cfg_id, fold_no, epoch, extra={"status": "early_stop"})
                    break

            save_checkpoint(cfg_id, fold_no, epoch, model, optimizer, scheduler,
                            best_val, best_epoch, epochs_no_improve, best_state,
                            extra={"params": params})
            save_progress(cfg_id, fold_no, epoch)

        if best_epoch == -1 or best_state is None:
            print("    ❌ Fold failed.")
            continue

        model.load_state_dict(best_state)

        # Final predictions
        model.eval()
        y_true_scaled_list, y_pred_scaled_list = [], []
        with torch.no_grad():
            for X_seq, y_t in val_loader:
                X_seq = X_seq.to(DEVICE)
                y_t   = y_t.to(DEVICE)
                y_hat = model(X_seq, A_dist, A_corr)
                y_true_scaled_list.append(y_t.cpu().numpy())
                y_pred_scaled_list.append(y_hat.cpu().numpy())

        y_true_scaled = np.concatenate(y_true_scaled_list, axis=0)
        y_pred_scaled = np.concatenate(y_pred_scaled_list, axis=0)

        y_true_orig = y_scaler.inverse_transform(y_true_scaled)
        y_pred_orig = y_scaler.inverse_transform(y_pred_scaled)

        mask_obs = np.isfinite(y_true_orig)
        y_true_vec = y_true_orig[mask_obs]
        y_pred_vec = y_pred_orig[mask_obs]

        if y_true_vec.size == 0:
            print("    ❌ No observed targets in val for metrics.")
            continue

        y_train_fold_orig = y_all_orig[train_start_idx:train_end_idx].reshape(-1)
        y_train_fold_orig = y_train_fold_orig[np.isfinite(y_train_fold_orig)]

        fold_mae  = mae(y_true_vec, y_pred_vec)
        fold_rmse = rmse(y_true_vec, y_pred_vec)
        fold_smape = smape(y_true_vec, y_pred_vec)
        fold_mase = mase(y_true_vec, y_pred_vec, y_train_fold_orig, m=12)

        print(f"    Fold {fold_no} MAE(£)={fold_mae:,.1f}, RMSE(£)={fold_rmse:,.1f}, "
              f"sMAPE={fold_smape:.3f}%, MASE={fold_mase:.3f}")

        # Update in-memory fold lists
        fold_mae_list.append(fold_mae)
        fold_rmse_list.append(fold_rmse)
        fold_smape_list.append(fold_smape)
        fold_mase_list.append(fold_mase)
        fold_count += 1
        done_folds.add(fold_no)

        # Persist fold result exactly once
        row = {
            "cfg_id": cfg_id,
            "fold_no": fold_no,
            "WINDOW": WINDOW,
            "HIDDEN_DIM": HIDDEN_DIM,
            "DROPOUT": DROPOUT,
            "LR": LR,
            "WEIGHT_DECAY": WD,
            "GATE_INIT": GATE_INIT,
            "K_DIST": K_DIST,
            "SIGMA_KM": SIGMA_KM,
            "K_CORR": K_CORR,
            "HUBER_BETA": HUBER_BETA,
            "MAE": float(fold_mae),
            "RMSE": float(fold_rmse),
            "sMAPE": float(fold_smape),
            "MASE": float(fold_mase),
        }
        append_fold_result(row, done_pairs)

        # Keep fr consistent during the run (so re-runs in same kernel are safe)
        fr = pd.concat([fr, pd.DataFrame([row])], ignore_index=True)

        save_progress(cfg_id, fold_no + 1, 1, extra={"status": "fold_done"})

    if fold_count == 0:
        print("  ❌ No valid folds for this config. Skipping.")
        save_progress(cfg_id + 1, 1, 1, extra={"status": "config_skipped"})
        continue

    cfg_result = {
        "cfg_id": cfg_id,
        "model_type": "TGCN_C1_GATED_IMPROVED",
        "WINDOW": WINDOW,
        "HIDDEN_DIM": HIDDEN_DIM,
        "DROPOUT": DROPOUT,
        "LR": LR,
        "WEIGHT_DECAY": WD,
        "GATE_INIT": GATE_INIT,
        "K_DIST": K_DIST,
        "SIGMA_KM": SIGMA_KM,
        "K_CORR": K_CORR,
        "HUBER_BETA": HUBER_BETA,
        "folds_used": fold_count,
        "MAE_mean":   float(np.mean(fold_mae_list)),
        "MAE_std":    float(np.std(fold_mae_list)),
        "RMSE_mean":  float(np.mean(fold_rmse_list)),
        "RMSE_std":   float(np.std(fold_rmse_list)),
        "sMAPE_mean": float(np.mean(fold_smape_list)),
        "sMAPE_std":  float(np.std(fold_smape_list)),
        "MASE_mean":  float(np.mean(fold_mase_list)),
        "MASE_std":   float(np.std(fold_mase_list)),
    }
    results.append(cfg_result)
    flush_results_partial(results)
    save_progress(cfg_id + 1, 1, 1, extra={"status": "config_done"})


Resume info: None

=== Config 1/288 ===
{'DROPOUT': 0.0, 'GATE_INIT': 0.85, 'HIDDEN_DIM': 64, 'HUBER_BETA': 1.0, 'K_CORR': 8, 'K_DIST': 8, 'LR': 0.001, 'SIGMA_KM': None, 'WEIGHT_DECAY': 0.0, 'WINDOW': 12}
  Fold 1: Train 2007-04–2014-09, Val 2014-10–2016-03
    Epoch 001 | trainHuber=0.2906 | valHuber=1.8014 | lr=1.0e-03 | gates(zr=0.850, h=0.850)
    Epoch 002 | trainHuber=0.2705 | valHuber=1.6528 | lr=1.0e-03 | gates(zr=0.850, h=0.851)
    Epoch 003 | trainHuber=0.2517 | valHuber=1.5059 | lr=1.0e-03 | gates(zr=0.850, h=0.851)
    Epoch 004 | trainHuber=0.2335 | valHuber=1.3868 | lr=1.0e-03 | gates(zr=0.850, h=0.851)
    Epoch 005 | trainHuber=0.2004 | valHuber=1.2609 | lr=1.0e-03 | gates(zr=0.849, h=0.852)
    Epoch 006 | trainHuber=0.2017 | valHuber=1.1163 | lr=1.0e-03 | gates(zr=0.849, h=0.852)
    Epoch 007 | trainHuber=0.1984 | valHuber=0.9699 | lr=1.0e-03 | gates(zr=0.849, h=0.852)
    Epoch 008 | trainHuber=0.1751 | valHuber=0.8239 | lr=1.0e-03 | gates(zr=0.848, h=0.853)
    Ep

C:\Users\slong\AppData\Local\Temp\ipykernel_14344\304172441.py:305: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  fr = pd.concat([fr, pd.DataFrame([row])], ignore_index=True)


    Epoch 002 | trainHuber=0.3094 | valHuber=2.2055 | lr=1.0e-03 | gates(zr=0.851, h=0.851)
    Epoch 003 | trainHuber=0.2781 | valHuber=2.0575 | lr=1.0e-03 | gates(zr=0.851, h=0.851)
    Epoch 004 | trainHuber=0.2828 | valHuber=1.8993 | lr=1.0e-03 | gates(zr=0.852, h=0.851)
    Epoch 005 | trainHuber=0.2286 | valHuber=1.7174 | lr=1.0e-03 | gates(zr=0.852, h=0.852)
    Epoch 006 | trainHuber=0.1925 | valHuber=1.5034 | lr=1.0e-03 | gates(zr=0.852, h=0.852)
    Epoch 007 | trainHuber=0.1428 | valHuber=1.2418 | lr=1.0e-03 | gates(zr=0.853, h=0.853)
    Epoch 008 | trainHuber=0.1084 | valHuber=0.9389 | lr=1.0e-03 | gates(zr=0.853, h=0.853)
    Epoch 009 | trainHuber=0.0901 | valHuber=0.6627 | lr=1.0e-03 | gates(zr=0.853, h=0.853)
    Epoch 010 | trainHuber=0.0945 | valHuber=0.5255 | lr=1.0e-03 | gates(zr=0.854, h=0.853)
    Epoch 011 | trainHuber=0.0976 | valHuber=0.5586 | lr=1.0e-03 | gates(zr=0.854, h=0.853)
    Epoch 012 | trainHuber=0.0920 | valHuber=0.6647 | lr=1.0e-03 | gates(zr=0.85

C:\Users\slong\AppData\Local\Temp\ipykernel_14344\304172441.py:305: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  fr = pd.concat([fr, pd.DataFrame([row])], ignore_index=True)


    Epoch 002 | trainHuber=0.3132 | valHuber=2.1076 | lr=5.0e-04 | gates(zr=0.850, h=0.850)
    Epoch 003 | trainHuber=0.2708 | valHuber=2.0289 | lr=5.0e-04 | gates(zr=0.849, h=0.849)
    Epoch 004 | trainHuber=0.2794 | valHuber=1.9508 | lr=5.0e-04 | gates(zr=0.849, h=0.849)
    Epoch 005 | trainHuber=0.2483 | valHuber=1.8724 | lr=5.0e-04 | gates(zr=0.849, h=0.849)
    Epoch 006 | trainHuber=0.2178 | valHuber=1.7926 | lr=5.0e-04 | gates(zr=0.849, h=0.849)
    Epoch 007 | trainHuber=0.2122 | valHuber=1.7113 | lr=5.0e-04 | gates(zr=0.849, h=0.849)
    Epoch 008 | trainHuber=0.1740 | valHuber=1.6238 | lr=5.0e-04 | gates(zr=0.849, h=0.849)
    Epoch 009 | trainHuber=0.1683 | valHuber=1.5287 | lr=5.0e-04 | gates(zr=0.849, h=0.849)
    Epoch 010 | trainHuber=0.1498 | valHuber=1.4241 | lr=5.0e-04 | gates(zr=0.849, h=0.849)
    Epoch 011 | trainHuber=0.1459 | valHuber=1.3073 | lr=5.0e-04 | gates(zr=0.849, h=0.849)
    Epoch 012 | trainHuber=0.1231 | valHuber=1.1757 | lr=5.0e-04 | gates(zr=0.84

...

    Epoch 001 | trainHuber=0.1415 | valHuber=0.4238 | lr=1.0e-04 | gates(zr=0.850, h=0.850)
    Epoch 002 | trainHuber=0.1457 | valHuber=0.4139 | lr=1.0e-04 | gates(zr=0.850, h=0.850)
    Epoch 003 | trainHuber=0.1304 | valHuber=0.4035 | lr=1.0e-04 | gates(zr=0.850, h=0.850)
    Epoch 004 | trainHuber=0.1211 | valHuber=0.3926 | lr=1.0e-04 | gates(zr=0.850, h=0.850)
    Epoch 005 | trainHuber=0.1054 | valHuber=0.3822 | lr=1.0e-04 | gates(zr=0.850, h=0.850)
    Epoch 006 | trainHuber=0.0977 | valHuber=0.3730 | lr=1.0e-04 | gates(zr=0.850, h=0.850)
    Epoch 007 | trainHuber=0.1071 | valHuber=0.3644 | lr=1.0e-04 | gates(zr=0.850, h=0.850)
    Epoch 008 | trainHuber=0.0970 | valHuber=0.3552 | lr=1.0e-04 | gates(zr=0.850, h=0.850)
    Epoch 009 | trainHuber=0.0852 | valHuber=0.3456 | lr=1.0e-04 | gates(zr=0.850, h=0.850)
    Epoch 010 | trainHuber=0.0715 | valHuber=0.3364 | lr=1.0e-04 | gates(zr=0.850, h=0.850)
    Epoch 011 | trainHuber=0.0718 | valHuber=0.3274 | lr=1.0e-04 | gates(zr=0.85

C:\Users\slong\AppData\Local\Temp\ipykernel_14344\304172441.py:305: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  fr = pd.concat([fr, pd.DataFrame([row])], ignore_index=True)


    Epoch 001 | trainHuber=0.0872 | valHuber=0.6190 | lr=1.0e-04 | gates(zr=0.850, h=0.850)
    Epoch 002 | trainHuber=0.0892 | valHuber=0.6043 | lr=1.0e-04 | gates(zr=0.850, h=0.850)
    Epoch 003 | trainHuber=0.0794 | valHuber=0.5884 | lr=1.0e-04 | gates(zr=0.850, h=0.850)
    Epoch 004 | trainHuber=0.0823 | valHuber=0.5750 | lr=1.0e-04 | gates(zr=0.850, h=0.850)
    Epoch 005 | trainHuber=0.0873 | valHuber=0.5615 | lr=1.0e-04 | gates(zr=0.850, h=0.850)
    Epoch 006 | trainHuber=0.0695 | valHuber=0.5467 | lr=1.0e-04 | gates(zr=0.850, h=0.850)
    Epoch 007 | trainHuber=0.0792 | valHuber=0.5318 | lr=1.0e-04 | gates(zr=0.850, h=0.850)
    Epoch 008 | trainHuber=0.0694 | valHuber=0.5159 | lr=1.0e-04 | gates(zr=0.850, h=0.850)
    Epoch 009 | trainHuber=0.0642 | valHuber=0.5012 | lr=1.0e-04 | gates(zr=0.850, h=0.850)
    Epoch 010 | trainHuber=0.0586 | valHuber=0.4881 | lr=1.0e-04 | gates(zr=0.850, h=0.850)
    Epoch 011 | trainHuber=0.0570 | valHuber=0.4756 | lr=1.0e-04 | gates(zr=0.85

C:\Users\slong\AppData\Local\Temp\ipykernel_14344\304172441.py:305: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  fr = pd.concat([fr, pd.DataFrame([row])], ignore_index=True)


    Epoch 001 | trainHuber=0.3118 | valHuber=1.8914 | lr=1.0e-04 | gates(zr=0.850, h=0.850)
    Epoch 002 | trainHuber=0.2744 | valHuber=1.8465 | lr=1.0e-04 | gates(zr=0.850, h=0.850)
    Epoch 003 | trainHuber=0.2866 | valHuber=1.8025 | lr=1.0e-04 | gates(zr=0.850, h=0.850)
    Epoch 004 | trainHuber=0.2709 | valHuber=1.7598 | lr=1.0e-04 | gates(zr=0.850, h=0.850)
    Epoch 005 | trainHuber=0.2499 | valHuber=1.7186 | lr=1.0e-04 | gates(zr=0.850, h=0.850)
    Epoch 006 | trainHuber=0.2741 | valHuber=1.6780 | lr=1.0e-04 | gates(zr=0.850, h=0.850)
    Epoch 007 | trainHuber=0.2456 | valHuber=1.6381 | lr=1.0e-04 | gates(zr=0.850, h=0.850)
    Epoch 008 | trainHuber=0.2484 | valHuber=1.5996 | lr=1.0e-04 | gates(zr=0.850, h=0.850)
    Epoch 009 | trainHuber=0.2472 | valHuber=1.5599 | lr=1.0e-04 | gates(zr=0.850, h=0.850)
    Epoch 010 | trainHuber=0.2412 | valHuber=1.5196 | lr=1.0e-04 | gates(zr=0.850, h=0.850)
    Epoch 011 | trainHuber=0.2377 | valHuber=1.4742 | lr=1.0e-04 | gates(zr=0.85

## Results

In [38]:
results_df = pd.DataFrame(results)
if not results_df.empty:
    results_df = results_df.sort_values("RMSE_mean").reset_index(drop=True)
    print("\n=== TOP CONFIGS BY MEAN VAL RMSE (ORIGINAL £) ===")
    print(results_df.head(15))
    out_path = "../../results/tgcn_c1_gated_improved_rollingcv_test.csv"
    results_df.to_csv(out_path, index=False)
    print(f"\nSaved tuning results to {out_path}")
else:
    print("\nNo successful configs to report.")


=== TOP CONFIGS BY MEAN VAL RMSE (ORIGINAL £) ===
    cfg_id              model_type  WINDOW  HIDDEN_DIM  DROPOUT      LR  \
0      116  TGCN_C1_GATED_IMPROVED      18          64      0.2  0.0003   
1      125  TGCN_C1_GATED_IMPROVED      12          64      0.2  0.0001   
2      124  TGCN_C1_GATED_IMPROVED      18          64      0.2  0.0001   
3       22  TGCN_C1_GATED_IMPROVED      18          64      0.0  0.0003   
4       12  TGCN_C1_GATED_IMPROVED      18          64      0.0  0.0005   
5       28  TGCN_C1_GATED_IMPROVED      18          64      0.0  0.0001   
6      128  TGCN_C1_GATED_IMPROVED      18          64      0.2  0.0001   
7       24  TGCN_C1_GATED_IMPROVED      18          64      0.0  0.0003   
8       14  TGCN_C1_GATED_IMPROVED      18          64      0.0  0.0005   
9       20  TGCN_C1_GATED_IMPROVED      18          64      0.0  0.0003   
10     216  TGCN_C1_GATED_IMPROVED      18          64      0.3  0.0003   
11     218  TGCN_C1_GATED_IMPROVED      18       